# Mobile App Rating Prediction using Machine Learning

Srinidhi S | MSc Data Science | 2303717673822046

## Objective

- This project aims to build a machine learning model that predicts the rating of mobile applications on the Google Play Store. By analyzing features such as number of installs, reviews, app size, price, and category, the model attempts to estimate how users are likely to rate an application.

- Predicting app ratings can help developers understand the factors that influence user satisfaction and improve application quality before release.

## Project Goals

- Clean the dataset by handling missing values and incorrect formats.
- Perform Exploratory Data Analysis (EDA) to understand feature relationships.
- Create visualizations to identify patterns in the data.
- Train machine learning models to predict app ratings.
- Evaluate model performance.
- Derive insights about factors influencing app ratings.

## Initial Data Exploration

In [43]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [55]:
df = pd.read_csv(r"D:\Mobile App Rating Prediction\data\googleplaystore.csv")
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [45]:
df.shape

(10841, 13)

In [46]:
df.columns

Index(['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type',
       'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver',
       'Android Ver'],
      dtype='object')

In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  object 
 1   Category        10841 non-null  object 
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  object 
 4   Size            10841 non-null  object 
 5   Installs        10841 non-null  object 
 6   Type            10840 non-null  object 
 7   Price           10841 non-null  object 
 8   Content Rating  10840 non-null  object 
 9   Genres          10841 non-null  object 
 10  Last Updated    10841 non-null  object 
 11  Current Ver     10833 non-null  object 
 12  Android Ver     10838 non-null  object 
dtypes: float64(1), object(12)
memory usage: 1.1+ MB


In [48]:
df.describe()

,Rating
count,9367.000000
mean,4.193338
std,0.537431
min,1.000000
25%,4.000000
50%,4.300000
75%,4.500000
max,19.000000


In [56]:
df.isnull().sum()

App                  0
Category             1
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       0
Genres               1
Last Updated         0
Current Ver          8
Android Ver          2
dtype: int64

## Data Preprocessing

### Reordering Columns

- For better clarity during model training, the target variable **Rating** is moved to the end of the dataset.

In [50]:
cols = [col for col in df.columns if col != "Rating"] + ["Rating"]
df = df[cols]

df.head()

,App,Category,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Rating
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up,4.1
1,Coloring book moana,ART_AND_DESIGN,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,3.9
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up,4.7
3,Sketch - Draw & Paint,ART_AND_DESIGN,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up,4.5
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up,4.3


### Handling Missing Values in Rating

- The **Rating** column is the target variable of this project. Rows where the rating is missing cannot be used for model training. Therefore, rows with missing ratings are removed.

In [67]:
df = df.dropna(subset=['Rating'])

df['Rating'].isnull().sum()

np.int64(0)

### Convert Reviews to Numeric

- The **Reviews** column represents the number of user reviews for each app. Since it is stored as a string, it must be converted into numeric format for analysis and modeling.

In [52]:
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')

### Cleaning the Installs Column

- The **Installs** column contains symbols such as commas and plus signs (e.g., 10,000+). These symbols must be removed so the column can be converted into numeric format.

In [57]:
df['Installs'] = df['Installs'].str.replace(',', '')
df['Installs'] = df['Installs'].str.replace('+', '')
df['Installs'] = df['Installs'].astype(int)

### Cleaning the Price Column

- The **Price** column contains dollar symbols (e.g., $4.99). The dollar symbol must be removed before converting the values into numeric format.

In [59]:
df['Price'] = df['Price'].str.replace('$', '')
df['Price'] = df['Price'].astype(float)

### Cleaning the Size Column

- The **Size** column contains values with units such as "M" (megabytes) and sometimes text like "Varies with device". The unit symbols are removed and values are converted to numeric format where possible.

In [61]:
df['Size'] = df['Size'].str.replace('M','')
df['Size'] = df['Size'].str.replace('k','')
df['Size'] = df['Size'].replace('Varies with device', np.nan)

df['Size'] = pd.to_numeric(df['Size'], errors='coerce')

### Converting Last Updated Column

- The **Last Updated** column represents the date when the application was last updated. This column is converted into a datetime format to make it easier to analyze time-based patterns.

In [64]:
df['Last Updated'] = pd.to_datetime(df['Last Updated'])

### Handling Remaining Missing Values

- Missing values in numerical columns are filled using the median value. The median is used because it is less sensitive to extreme values.

In [ ]:
df['Size'].fillna(df['Size'].median(), inplace=True)
df['Reviews'].fillna(df['Reviews'].median(), inplace=True)

In [73]:
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19.0,10000,Free,0.0,Everyone,Art & Design,2018-01-07,1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,2018-01-15,2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7,5000000,Free,0.0,Everyone,Art & Design,2018-08-01,1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25.0,50000000,Free,0.0,Teen,Art & Design,2018-06-08,Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8,100000,Free,0.0,Everyone,Art & Design;Creativity,2018-06-20,1.1,4.4 and up


In [ ]:
df = df.dropna()
df.isnull().sum()

App               0
Category          0
Rating            0
Reviews           0
Size              0
Installs          0
Type              0
Price             0
Content Rating    0
Genres            0
Last Updated      0
Current Ver       0
Android Ver       0
dtype: int64

### Final Dataset Shape After Cleaning

- After removing rows with missing values and cleaning the dataset, we check the final dataset shape.

In [72]:
df.shape

(9360, 13)

## Observations After Data Preprocessing

After completing the preprocessing steps, the dataset has been cleaned and prepared for analysis and machine learning. The following key changes were made:

- Some missing numeric values were handled by **replacing them with the median**, ensuring minimal impact from outliers.
- A small number of remaining rows with missing categorical values were **removed from the dataset**.
- Columns such as **Reviews, Installs, Price, and Size** were converted into numeric format so they can be used in machine learning models.
- Symbols such as **commas (+), dollar signs ($), and size units (M, k)** were removed to standardize numeric values.
- The **Last Updated** column was converted into datetime format for better handling of date information.
- The **Rating** column was moved to the end of the dataset to clearly represent it as the target variable.

After preprocessing, the dataset is now clean, consistent, and suitable for performing exploratory data analysis and training machine learning models.